In [4]:
pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_core-1.6.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pypdf-6.17.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.12.2-py3-none-any.whl.metadata (22 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgr

In [6]:
from langchain_core.documents import Document

In [7]:
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_17128\3383052197.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [8]:
# document = loader.load()

In [9]:
# document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [12]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/resource.pdf")

# document1 = pdf_loader.load()

# document1

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-08-29T16:33:21+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

In [14]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [20]:
# data => documents

def load_all_pdf():
    folder_path = 'data/pdfs'
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total_pdf", num_docs)
    print("total_pages", len(all_docs))
    return all_docs

In [21]:
all_pds_documents = load_all_pdf()

total_pdf 2
total_pages 32


In [22]:
!pip install langchain_text_splitters

In [23]:
# documents => chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):

    text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitters.split_documents(documents)
    return chunked_docs

In [24]:
chunks = split_doc(all_pds_documents)

In [25]:
len(chunks)

321

In [26]:
from sentence_transformers import SentenceTransformer

### Embedding

In [31]:
class EmbeddingManager():
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("Loading Model.....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimension:", self.model.get_sentence_embedding_dimension())


    def generate_embedding(self, text):
        embedding = self.model.encode(text, show_progress_bar=True)
        print("embedding shape", embedding.shape)
        return embedding

In [32]:
embedding_manager = EmbeddingManager()

Loading Model..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension: 384


C:\Users\Mohd Zaid\AppData\Local\Temp\ipykernel_17128\2330007166.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimension:", self.model.get_sentence_embedding_dimension())


### Vector DB

In [36]:
import chromadb
import uuid

In [37]:
class VectorStoreManager:

    def __init__(
        self,
        persist_directory="data/vector",
        collection_name="pdf_documents",
    ):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None  # Fixed typo: 'colection' -> 'collection'
        self.client = None

        self.initialize_store()  # Matches method name below

    def initialize_store(self):  # Removed leading underscore to match call
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embedding in RAG"
            },
        )

        # Indented inside initialize_store method
        print(
            "Initialize the vector store with collection:", self.collection_name
        )
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError(
                "num of documents does not match num of embeddings"
            )

        ids = []
        all_metadata = []
        documents_content = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            # Fixed: Append text content of current doc, not the whole 'documents' list
            documents_content.append(doc.page_content)

            # Convert embedding (numpy array or tensor) to list
            embedding_list.append(
                embedding.tolist()
                if hasattr(embedding, "tolist")
                else embedding
            )

        # Moved outside the for-loop to execute as a single batch operation
        self.collection.add(
            ids=ids,
            documents=documents_content,
            metadatas=all_metadata,  # Note: Chroma parameter name is 'metadatas'
            embeddings=embedding_list,
        )

        print("total documents added in vector store =", len(documents_content))
        print("docs in collection:", self.collection.count())

In [38]:
vector_store = VectorStoreManager()

Initialize the vector store with collection: pdf_documents
docs in collection: 0


In [45]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape (321, 384)
total documents added in vector store = 321
docs in collection: 642


In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
class RAGRetriever:

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embedding([query])[
            0
        ]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()], n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1,
                    })

            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("no documents found")

        return retrieved_docs

In [48]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [49]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 5 documents


[{'id': 'doc_47e2c56e-10bf-4d21-90c4-c8b9d6210c88',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'trapped': '/False',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25',
   'content_length': 288,
   'source': 'data/pdfs\\research2.pdf',
   'title': '',
   'subject': '',
   'keywords': '',
   'total_pages': 21,
   'page_label': '1',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'doc_index': 11,
   'creator': 'LaTeX with hyperref',
   'page': 0,
   'author': '',
   'moddate': '2024-03-28T00:54:45+00:00'},
  'distance': 0.46291494369506836,
  'similarity_score': 0.5370850563049316,
  'rank': 1},
 {'id': 'doc_24d04988

In [50]:
rag_retriever.retrieve("What are encoder and decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 5 documents


[{'id': 'doc_7269e0d0-df0a-452b-8324-6cd659d8dc62',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\nsaklainreza95@gmail.com',
  'metadata': {'lastpage': '6008',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'published': '2017',
   'eventtype': 'Poster',
   'date': '2017',
   'title': 'Attention is All you Need',
   'publisher': 'Curran Associates, Inc.',
   'content_length': 136,
   'producer': 'pdfcpu v0.12.1 dev',
   'source': 'data/pdfs\\resource.pdf',
   'total_pages': 11,
   'created': '2017',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also 

### Integrate with LLM

In [51]:
api_key = ''

In [52]:
!pip install langchain-groq

  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_groq-1.1.3-py3-none-any.whl (20 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [54]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = api_key,
    model = "openai/gpt-oss-120b",
    temperature = 0.1,
    max_tokens = 1024
)

Task was destroyed but it is pending!
task: <Task pending name='Task-743' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\Mohd Zaid\anaconda3\envs\data-science\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-744' coro=<Kernel.shell_main() running at C:\Users\Mohd Zaid\anaconda3\envs\data-science\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\Mohd Zaid\anaconda3\envs\data-science\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
<string>:6: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
Task was destroyed but it is pending!
task: <Task pending name='Task-744' coro=<Kernel.shell_main() running at C:\Users\Mohd Zaid\anaconda3\envs\data-science\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


In [57]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = (
        "\n".join([doc["document"] for doc in results]) if results else ""
    )

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
    Context: {context}
    Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)])  # expecting a list as prompt
    return response.content

In [58]:
# Usage
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 3 documents


In [60]:
print(answer)

**RAG (Retrieval‑Augmented Generation)** is a hybrid AI architecture that combines two complementary capabilities:

| Component | What it does | How it contributes to RAG |
|-----------|--------------|---------------------------|
| **Retriever** | Searches an external knowledge source (e.g., a document corpus, database, or the web) to fetch passages that are relevant to the current input query. | Supplies factual, up‑to‑date information that the language model alone might not know or might hallucinate. |
| **Generator** | A large language model (LLM) that takes the retrieved passages (often concatenated with the original prompt) and produces a natural‑language response. | Uses the retrieved evidence to ground its output, improving factual accuracy and allowing the system to answer questions beyond the model’s internal training data. |

### Why RAG matters
- **Enhanced factuality** – By grounding generation in real documents, RAG reduces hallucinations that pure LLMs often produce.
- **